# Shortcut Test Set: Confusion Matrices and Evaluation Metrics

Compares each model on two versions of the **same** held-out test images:

- **Shortcut present**: the original images (true colours).
- **Shortcut absent**: the same images with the R/G/B channels permuted by a random non-identity permutation (one fixed permutation per image). Shape and texture are unchanged, the colour cue is wrong, and nothing else is altered.

The set is built from the CIFAR-10 **test split only**, so it is separate from training data. Classes: bird, cat, deer, dog, frog, horse (1,000 images each).

**Before running:** edit the paths in the next cell.

In [ ]:
# ---- CONFIG: edit these paths ----
MODEL_PATHS = {
    "run1_original": "run1_original_model.keras",   # path to the model saved by your Colab run
    "run4_heavy":    "run4_heavy_model.keras",
}
NPZ_PATH = "shortcut_test_set.npz"   # saved test set; if the file does not exist it is built below
SEED = 2026                          # seed for the colour permutations

In [ ]:
import os, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from tensorflow import keras
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, log_loss

ANIMALS = ["bird", "cat", "deer", "dog", "frog", "horse"]
N = len(ANIMALS)
PERMS = [(0, 2, 1), (1, 0, 2), (1, 2, 0), (2, 0, 1), (2, 1, 0)]   # the 5 non-identity R/G/B permutations

## 1. Build (or load) the shortcut test set

In [ ]:
def load_cifar_animal_test():
    """CIFAR-10 test split, animal classes only. Labels remapped 2-7 -> 0-5 (bird..horse), pixels in [0, 1]."""
    try:
        (_, _), (x, y) = keras.datasets.cifar10.load_data()
    except Exception as e:
        print("keras download failed (", e, ") - trying Hugging Face instead")
        from datasets import load_dataset
        hf = load_dataset("uoft-cs/cifar10")["test"]
        x = np.stack([np.array(im) for im in hf["img"]]); y = np.array(hf["label"])
    y = np.asarray(y).ravel(); keep = np.isin(y, [2, 3, 4, 5, 6, 7])
    return x[keep].astype("float32") / 255.0, (y[keep] - 2).astype("int64")

if os.path.exists(NPZ_PATH):
    d = np.load(NPZ_PATH)
    images_present, images_absent, labels = d["images_present"], d["images_absent"], d["labels"]
    print("Loaded", NPZ_PATH)
else:
    images_present, labels = load_cifar_animal_test()
    rng = np.random.default_rng(SEED)
    perm_ids = rng.integers(0, len(PERMS), size=len(images_present))      # one fixed permutation per image
    images_absent = np.stack([images_present[i][..., list(PERMS[p])] for i, p in enumerate(perm_ids)])
    np.savez_compressed(NPZ_PATH, images_present=images_present, images_absent=images_absent,
                        labels=labels, perm_ids=perm_ids, class_names=np.array(ANIMALS))
    with open("shortcut_test_set_how_built.json", "w") as f:
        json.dump({
            "source": "CIFAR-10 test split, animal classes only",
            "n_images": int(len(labels)),
            "images_per_class": {ANIMALS[c]: int((labels == c).sum()) for c in range(N)},
            "shortcut": "colour (typical class colours)",
            "present": "unmodified test images",
            "absent": "R/G/B channels permuted by a random non-identity permutation (one per image); nothing else changed",
            "permutations_used": {str(PERMS[k]): int((perm_ids == k).sum()) for k in range(len(PERMS))},
            "seed": SEED,
            "kept_separate_from_training": "built from the held-out test split only",
        }, f, indent=2)
    print("Built and saved", NPZ_PATH, "and shortcut_test_set_how_built.json")

assert images_present.shape == images_absent.shape and len(labels) == len(images_present)
assert not np.allclose(images_present, images_absent)
assert np.allclose(images_present.mean(axis=-1), images_absent.mean(axis=-1), atol=1e-5)  # grey-level content unchanged
print(images_present.shape, np.bincount(labels))

In [ ]:
# Look at a few pairs: top row = shortcut present, bottom row = shortcut absent
idx = np.random.default_rng(0).choice(len(labels), 8, replace=False)
fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
for j, i in enumerate(idx):
    axes[0, j].imshow(images_present[i]); axes[0, j].set_title(ANIMALS[labels[i]], fontsize=9)
    axes[1, j].imshow(images_absent[i])
    for r in (0, 1): axes[r, j].axis("off")
plt.suptitle("Shortcut present (top) vs absent (bottom)"); plt.tight_layout(); plt.show()

## 2. Evaluate each model on both sets

In [ ]:
def evaluate(model, images, y):
    probs = model.predict(images, verbose=0, batch_size=500)
    pred = probs.argmax(axis=1)
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=range(N), zero_division=0)
    cm = confusion_matrix(y, pred, labels=range(N))
    return dict(cm=cm, cm_norm=cm / cm.sum(axis=1, keepdims=True), precision=p, recall=r, f1=f,
                acc=(pred == y).mean(), macro_f1=f.mean(), loss=log_loss(y, probs, labels=range(N)),
                conf=probs.max(axis=1).mean(), pred=pred)

results = {}
for name, path in MODEL_PATHS.items():
    model = keras.models.load_model(path)
    results[name] = {"present": evaluate(model, images_present, labels),
                     "absent":  evaluate(model, images_absent, labels)}
    print(name, f"present {results[name]['present']['acc']:.1%} | absent {results[name]['absent']['acc']:.1%}")

## 3. Confusion matrices: one per test set, side by side, plus the difference

In [ ]:
for name, R in results.items():
    A, B = R["present"], R["absent"]
    fig, axes = plt.subplots(1, 3, figsize=(19, 5.6))
    for ax, (title, S) in zip(axes[:2], (("Shortcut present", A), ("Shortcut absent", B))):
        sns.heatmap(S["cm_norm"], annot=S["cm"], fmt="d", cmap="Blues", vmin=0, vmax=1, cbar=False,
                    xticklabels=ANIMALS, yticklabels=ANIMALS, ax=ax)
        ax.set_title(f"{title}\naccuracy {S['acc']:.1%}"); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    sns.heatmap(A["cm_norm"] - B["cm_norm"], annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-0.6, vmax=0.6,
                xticklabels=ANIMALS, yticklabels=ANIMALS, ax=axes[2])
    axes[2].set_title("Present minus absent (share of each true class)\nred = more when shortcut present")
    axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("True")
    fig.suptitle(name, y=1.02, fontsize=13); plt.tight_layout()
    plt.savefig(f"confusion_matrices_{name}.png", dpi=130, bbox_inches="tight"); plt.show()

## 4. Evaluation metrics

In [ ]:
rows, per_class = [], []
for name, R in results.items():
    for st in ("present", "absent"):
        S = R[st]
        rows.append(dict(model=name, set=st, accuracy=S["acc"], macro_f1=S["macro_f1"], log_loss=S["loss"], mean_confidence=S["conf"]))
        for i, c in enumerate(ANIMALS):
            per_class.append(dict(model=name, set=st, cls=c, precision=S["precision"][i], recall=S["recall"][i], f1=S["f1"][i]))
overall = pd.DataFrame(rows); per_class = pd.DataFrame(per_class)
overall.round(4).to_csv("shortcut_metrics_overall.csv", index=False)
per_class.round(4).to_csv("shortcut_metrics_per_class.csv", index=False)
display(overall.round(3))

# per-class recall, present vs absent, with the gap
recall = per_class.pivot_table(index="cls", columns=["model", "set"], values="recall").reindex(ANIMALS)
for name in results:
    recall[(name, "gap (present - absent)")] = recall[(name, "present")] - recall[(name, "absent")]
display(recall.round(3))

## 5. Observations (generated from the numbers above, so they update when you re-run)

In [ ]:
for name, R in results.items():
    A, B = R["present"], R["absent"]
    gap = A["acc"] - B["acc"]
    rec_gap = pd.Series(A["recall"] - B["recall"], index=ANIMALS)
    shift = B["cm_norm"] - A["cm_norm"]
    errs = sorted([(ANIMALS[i], ANIMALS[j], shift[i, j]) for i in range(N) for j in range(N) if i != j], key=lambda t: -t[2])[:3]
    share = np.bincount(B["pred"], minlength=N) / len(B["pred"])
    print(f"{name}")
    print(f"  accuracy present {A['acc']:.1%}, absent {B['acc']:.1%}  (gap {gap*100:+.1f} points)")
    print(f"  biggest recall losses when colour is scrambled: " + ", ".join(f"{c} {rec_gap[c]*100:+.1f}" for c in rec_gap.abs().sort_values(ascending=False).index[:3]))
    print(f"  errors that grow most when the shortcut is removed: " + ", ".join(f"{t}->{p} {d*100:+.1f} pts" for t, p, d in errs))
    print(f"  most-predicted class on the absent set: {ANIMALS[share.argmax()]} ({share.max():.0%} of predictions; even would be {1/N:.0%})")
    print(f"  mean confidence present {A['conf']:.3f}, absent {B['conf']:.3f}\n")

## Notes

- Reading the results: a large accuracy gap (present minus absent) means the model relies on colour; a gap near zero means it does not.
- A channel shuffle also makes images look unnatural, so part of a drop may come from unfamiliar images as well as lost colour cues. A model trained on colour-swapped images (such as `run4_heavy`) shows how large that effect is.
- Colour is only one shortcut. This notebook does not test backgrounds or textures.
- The set uses the 6,000 animal images from the CIFAR-10 test split. Reported accuracies can differ slightly from your Colab notebook if the images were loaded from a different copy of the dataset (for example JPEG-compressed copies score lower).